# Copper price prediction — LSTM & SA-LSTM (Kaggle)

**Bám theo:** Chen et al. (2023), *PLOS ONE*, DOI: [10.1371/journal.pone.0285631](https://doi.org/10.1371/journal.pone.0285631). Dữ liệu **Yahoo Finance, 2006-01-01 đến hết 2026-11-01** (khác dữ liệu Investing.com 1990–2009 của tác giả).

**Thứ tự:** tải 8 biến ứng viên + giá đồng → chia thời gian → Spearman trên tập fit của TRAIN → chọn biến → scaler fit tập fit của TRAIN → sequences với nhãn là ngày kế tiếp → validation tách từ TRAIN → LSTM cơ sở và SA-LSTM đã tối ưu → test cuối cùng. Ba tỷ lệ train/test 90/10, 92.5/7.5, 95/5 theo bài báo.

**Lưu ý khoa học:** 39/111 neuron và LR=5e-5 là **hyperparameters**, không phải trọng số đã học. Mặc định `SA_MODE='paper_preset'` dùng thông số mà SA tìm được trong bài; `SA_MODE='rerun'` thực sự chạy SA mới (chỉ validation, tốn tài nguyên). Không được gọi preset là kết quả tự chạy SA trên dữ liệu mới. Bài gốc không mô tả đầy đủ lookback/validation; dùng 30 phiên theo notebook gốc, và 10% cuối TRAIN làm validation. Dữ liệu đầu vào là 3 chỉ báo được chọn bằng Spearman, **không đưa giá đồng cùng ngày làm feature**. Dùng quá khứ dự báo ngày giao dịch tiếp theo.

In [ ]:
%pip -q install -U yfinance

In [ ]:
import os, random, warnings, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception: pass
START, END_EXCLUSIVE = '2006-01-01', '2026-11-02'
SEQ_LENGTH = 30
TRAIN_RATIOS = [0.90, 0.925, 0.95]
EPOCHS = 100
BATCH = 64
SA_MODE = 'paper_preset'    # 'paper_preset' (default) | 'rerun'
SPEARMAN_CUTOFF = 0.80    # bài báo dùng >0.8 (tương quan dương)
PAPER_FEATURES = ['WTI','Gold','Silver']
# Nếu dữ liệu mới không còn chỉ báo nào >0.8, chọn top 3 dựa trên TRAIN và ghi rõ là thích nghi
print('TensorFlow:', tf.__version__, '| yfinance:', yf.__version__)

## 1. Dữ liệu — 8 chỉ báo của Table 1 và target

Dow Jones, Nasdaq, Dollar Index, US Treasury 10Y yield, Gold, Silver, WTI Oil, **Copper Volume**; target: **Copper Close**. `^TNX` thường đã được Yahoo niêm yết theo %, chỉ chia 10 nếu dữ liệu ở thang 10×. Dùng **inner join theo ngày** và xóa ngày có dữ liệu khuyết, đúng quy tắc bài báo. Đóng cửa hợp đồng tương lai Yahoo là proxy, không hoàn toàn đồng nhất chuỗi lịch sử Investing.

In [ ]:
TICKERS = {
    'Copper': ['HG=F'], 'Dow_Jones':['^DJI'], 'Nasdaq':['^IXIC'],
    'Dollar_Index':['DX-Y.NYB','DX=F'], 'TBill_10Y':['^TNX'],
    'Gold':['GC=F'], 'Silver':['SI=F'], 'WTI':['CL=F']
}

def fetch_one(ticker):
    # Tải từng ticker để tránh MultiIndex và sự cố batch Yahoo
    df = yf.download(ticker, start=START, end=END_EXCLUSIVE,
                     auto_adjust=False, progress=False, threads=False)
    if df.empty: raise ValueError(f'Yahoo trả về rỗng: {ticker}')
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    if 'Close' not in df: raise ValueError(f'Thiếu Close: {ticker}')
    df.index = pd.to_datetime(df.index).tz_localize(None).normalize()
    return df.loc[~df.index.duplicated(keep='last')].sort_index()

sources = {}; series = {}
for name, candidates in TICKERS.items():
    last_err = None
    for ticker in candidates:
        try:
            frame = fetch_one(ticker)
            if frame['Close'].notna().sum() < 1000:
                raise ValueError(f'Không đủ lịch sử cho {ticker}')
            sources[name] = ticker
            series[name] = pd.to_numeric(frame['Close'], errors='coerce').rename(name)
            if name == 'Copper':
                series['Copper_Volume'] = pd.to_numeric(frame['Volume'], errors='coerce').rename('Copper_Volume')
            print(f'{name:15s} {ticker:10s} {len(frame):5d} rows | {frame.index.min().date()} → {frame.index.max().date()}')
            break
        except Exception as ex:
            last_err = ex
            print(f'WARN {name}: {ticker} không tải được: {str(ex)[:130]}')
    else:
        raise RuntimeError(f'Không lấy được {name}, đã thử {candidates}. Bật Kaggle Internet và Run All lại.') from last_err

raw = pd.concat(series.values(), axis=1, join='inner').sort_index()
raw = raw.loc[(raw.index >= START) & (raw.index < END_EXCLUSIVE)].copy()
# Yahoo ^TNX currently quotes yield directly in % (~4-5), some legacy feeds use 10x.
if raw['TBill_10Y'].median() > 15:
    raw['TBill_10Y'] = raw['TBill_10Y'] / 10.0
missing = raw.isna().sum().rename('missing_rows_before_drop')
print('Missing per column:', missing.to_string())
raw = raw.replace([np.inf,-np.inf], np.nan).dropna(how='any')
raw = raw.loc[~raw.index.duplicated(keep='last')]
assert raw.index.is_monotonic_increasing and raw.index.is_unique
assert len(raw) > 1000, f'Chỉ {len(raw)} phiên hoàn chỉnh; kiểm tra nguồn dữ liệu'
assert len(raw.columns) == 9 and raw.columns.is_unique
print('Dữ liệu hoàn chỉnh:', raw.shape, '→', raw.index.min().date(), raw.index.max().date())
print('Ticker mapping:', sources)
display(raw.head(3)); display(raw.tail(3))

In [ ]:
# Lưu dữ liệu đã tải và làm sạch thành CSV (gồm Date + target + 8 biến kinh tế)
from pathlib import Path

output_dir = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
csv_path = output_dir / 'copper_8_indicators_2006_2026.csv'
raw.to_csv(csv_path, index=True, index_label='Date', encoding='utf-8-sig')
print(f'Đã lưu {len(raw):,} dòng, {len(raw.columns)} cột dữ liệu vào: {csv_path}')
print('Kaggle: tải file tại Output > /kaggle/working/')

## 2. Spearman chỉ trên tập fit — ngăn leakage

Bài gốc dùng Spearman do phân phối không chuẩn; `r > 0.8` (không dùng trị tuyệt đối). Do thay giai đoạn nghiên cứu, biến được chọn có thể khác WTI/Gold/Silver. Để tránh thông tin tương lai, **mỗi tỷ lệ train/test tính tương quan lại từ phần fit của TRAIN (không dùng validation/test)**. Nếu không có biến nào đạt ngưỡng, chọn 3 biến tương quan dương mạnh nhất trên tập fit (ngoại lệ được in ra), không tuyên bố đã tái hiện nguyên xi feature selection.

In [ ]:
CANDIDATES = ['Dow_Jones','Nasdaq','Dollar_Index','TBill_10Y','Gold','Silver','WTI','Copper_Volume']

def get_split_context(ratio):
    split_row = int(len(raw)*ratio) # chia THEO NGÀY trước bất kỳ thống kê học nào
    val_start = int(split_row * .90)
    fit_raw = raw.iloc[:val_start]  # do not leak validation into SA search
    corr = fit_raw[CANDIDATES].corrwith(fit_raw['Copper'], method='spearman').sort_values(ascending=False)
    chosen = corr[corr > SPEARMAN_CUTOFF].index.tolist()
    if not chosen:
        chosen = corr.head(3).index.tolist()
        warnings.warn(f'{ratio:.1%}: không có Spearman > {SPEARMAN_CUTOFF}; fallback top 3 trên TRAIN!')
    xscaler = MinMaxScaler().fit(fit_raw[chosen].to_numpy())
    yscaler = MinMaxScaler().fit(fit_raw[['Copper']].to_numpy())
    # Transform cả tập bằng thông số học CHỈ từ TRAIN. Test có thể ra ngoài [0,1], là hợp lệ.
    x_all = xscaler.transform(raw[chosen].to_numpy()).astype('float32')
    y_all = yscaler.transform(raw[['Copper']].to_numpy()).astype('float32').reshape(-1)
    # test label index >= split_row; x của label t chỉ dùng các dòng t-SEQ_LENGTH...t-1
    # val nằm cuối TRAIN; split val theo TARGET index, không dựa vào window index
    assert val_start > SEQ_LENGTH
    def windows(label_start, label_end):
        X = np.stack([x_all[t-SEQ_LENGTH:t] for t in range(label_start, label_end)])
        y = y_all[label_start:label_end]
        dates = raw.index[label_start:label_end]
        return X, y, dates
    Xtr,ytr,dtr = windows(SEQ_LENGTH,val_start)
    Xva,yva,dva = windows(val_start,split_row)
    Xte,yte,dte = windows(split_row,len(raw))
    assert dtr.max() < dva.min() < dte.min()
    assert dte.min() == raw.index[split_row]
    assert Xtr.shape[-1] == len(chosen)
    return dict(ratio=ratio, corr=corr, chosen=chosen, scaler=yscaler,
                train=(Xtr,ytr,dtr), val=(Xva,yva,dva), test=(Xte,yte,dte), split=split_row)

contexts = [get_split_context(r) for r in TRAIN_RATIOS]
for ctx in contexts:
    print(f"train={ctx['ratio']:.1%} | features={ctx['chosen']} | "
          f"train/val/test = {[len(ctx[k][1]) for k in ['train','val','test']]}")
    display(ctx['corr'].rename('Spearman_vs_Copper_TRAIN').to_frame().T.round(3))
# Báo cáo Spearman tính lại cho toàn bộ 8 ứng viên; KHÔNG dùng test để chọn biến.
fig, axes = plt.subplots(1,3,figsize=(15,4),sharey=True)
for ax,ctx in zip(axes,contexts):
    ctx['corr'].sort_values().plot.barh(ax=ax)
    ax.axvline(.8,color='red',ls='--'); ax.set_title(f"Train {ctx['ratio']:.1%}"); ax.set_xlim(-1,1)
plt.tight_layout(); plt.show()

## 3. Kiến trúc và huấn luyện

Theo Fig.12/Table 4 bài báo: LSTM(39) → LSTM(111) → Dense(32) → Dropout(0.2) → Dense(1); SA-LSTM `LR=5e-5`, `epochs=100`, `batch=64`. Baseline cùng kiến trúc nhưng chọn cố định `32/32`, `LR=1e-3`, **không tối ưu** (lựa chọn của đồ án, không phải hyperparameters baseline do bài báo công bố). Hai mô hình train độc lập từ random initialization, trên cùng dữ liệu/tỷ lệ, không dùng test làm validation.

In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

PAPER_SA = {'lr': 0.00005, 'units1':39, 'units2':111}
BASELINE = {'lr': 0.001, 'units1':32, 'units2':32}

def build_lstm(n_features, hp):
    model = Sequential([
        Input(shape=(SEQ_LENGTH,n_features)),
        LSTM(int(hp['units1']),return_sequences=True),
        LSTM(int(hp['units2'])),
        Dense(32,activation='relu'), Dropout(0.2), Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=float(hp['lr'])),loss='mse')
    return model

def fit_and_evaluate(ctx,hp,name,epochs=EPOCHS):
    tf.keras.backend.clear_session(); gc.collect()
    tf.keras.utils.set_random_seed(SEED)
    Xtr,ytr,_=ctx['train']; Xva,yva,_=ctx['val']; Xte,yte,dte=ctx['test']
    net=build_lstm(Xtr.shape[-1],hp)
    history=net.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=epochs,
                    batch_size=BATCH,shuffle=False,verbose=0)
    pred_scaled=net.predict(Xte,batch_size=BATCH,verbose=0)
    true=ctx['scaler'].inverse_transform(yte.reshape(-1,1)).reshape(-1)
    pred=ctx['scaler'].inverse_transform(pred_scaled).reshape(-1)
    out=dict(model=name,ratio=ctx['ratio'],features=', '.join(ctx['chosen']),
             MSE_scaled=mean_squared_error(yte,pred_scaled.reshape(-1)),
             MSE_USD2=mean_squared_error(true,pred),
             RMSE_USD=np.sqrt(mean_squared_error(true,pred)),
             MAE_USD=mean_absolute_error(true,pred),
             lr=hp['lr'],units1=hp['units1'],units2=hp['units2'],
             test_start=str(dte.min().date()),test_end=str(dte.max().date()),n_test=len(dte))
    return out,dict(date=dte,true=true,pred=pred,history=history.history),net

## 4. SA tùy chọn (mặc định giữ nguyên tham số đã tối ưu trong bài)

`SA_MODE='rerun'`: chạy Simulated Annealing tối ưu learning rate và số neuron trên **validation**, tuyệt đối không đọc test; quy tắc Metropolis `exp(-ΔE/T)` và cooling `0.95`. Giới hạn ngân sách mặc định 12 bước × 12 epochs để khả thi trên Kaggle; **không tương đương 650 scenarios / 100 epochs của bài báo**. Để tái hiện nguyên gốc nghiêm ngặt, tăng số bước/epochs (rất tốn GPU). Mặc định **không tìm lại** vì người dùng muốn bảo toàn bộ hyperparameters 39/111/5e-5.

In [ ]:
def sa_search(ctx, steps=12, trial_epochs=12, t_initial=1., cooling=.95):
    Xtr,ytr,_=ctx['train']; Xva,yva,_=ctx['val']
    rng=np.random.default_rng(SEED)
    def score(hp):
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
        net=build_lstm(Xtr.shape[-1],hp)
        net.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=trial_epochs,
                batch_size=BATCH,shuffle=False,verbose=0)
        loss=float(net.evaluate(Xva,yva,verbose=0))
        del net; gc.collect()
        return loss
    current=dict(PAPER_SA); current_loss=score(current)
    best=dict(current); best_loss=current_loss; T=t_initial
    logs=[{'step':0,'T':T,'val_mse':current_loss,**current}]
    for i in range(1,steps+1):
        # Perturb log learning rate and neuron count, then clip to feasible range
        candidate={
            'lr':float(np.clip(current['lr']*np.exp(rng.uniform(-1,1)*T*2),1e-6,0.005)),
            'units1':int(np.clip(round(current['units1']+rng.uniform(-1,1)*T*96),8,128)),
            'units2':int(np.clip(round(current['units2']+rng.uniform(-1,1)*T*96),8,128))}
        new_loss=score(candidate); delta=new_loss-current_loss
        if delta <=0 or rng.random() < np.exp(-delta/max(T*0.01,1e-10)):
            current,current_loss=candidate,new_loss
        if new_loss<best_loss: best,best_loss=dict(candidate),new_loss
        logs.append({'step':i,'T':T,'val_mse':new_loss,**candidate})
        T*=cooling
        print(f'SA {i}/{steps} | val MSE={new_loss:.6f} | best={best_loss:.6f}')
    return best,pd.DataFrame(logs)

SA_HP_BY_RATIO={}
if SA_MODE == 'rerun':
    for ctx in contexts:
        hp,log=sa_search(ctx)
        SA_HP_BY_RATIO[ctx['ratio']]=hp
        display(log.tail())
else:
    assert SA_MODE == 'paper_preset', 'SA_MODE must be paper_preset or rerun'
    SA_HP_BY_RATIO={ctx['ratio']:dict(PAPER_SA) for ctx in contexts}
print('SA mode:',SA_MODE,'| SA hyperparams:',SA_HP_BY_RATIO)

## 5. So sánh cùng các tỷ lệ 90%, 92.5%, 95%

100 epochs mỗi mô hình như bài báo; không early stopping để giữ đúng số epoch. Validation chỉ để theo dõi (và để chạy SA khi chọn `rerun`). Test dùng **một lần** sau train. MSE trên thang [0,1] và trên USD² là hai đại lượng khác nhau, không trộn lẫn.

In [ ]:
results=[]; traces={}
for ctx in contexts:
    ratio=ctx['ratio']; print(f'\n=== TRAIN RATIO {ratio:.1%} ===')
    for name,hp in [('LSTM baseline',BASELINE),('SA-LSTM ('+SA_MODE+')',SA_HP_BY_RATIO[ratio])]:
        row,trace,net=fit_and_evaluate(ctx,hp,name)
        results.append(row); traces[(ratio,name)]=trace
        print(name,'MSE scaled=',round(row['MSE_scaled'],6),'MSE USD²=',round(row['MSE_USD2'],6))
        del net; gc.collect()
report=pd.DataFrame(results)
display(report.round(6))
report.to_csv('/kaggle/working/copper_lstm_comparison.csv' if os.path.isdir('/kaggle/working') else 'copper_lstm_comparison.csv',index=False)

In [ ]:
fig,axes=plt.subplots(3,1,figsize=(15,12))
for ax,ctx in zip(axes,contexts):
    ratio=ctx['ratio']; names=['LSTM baseline','SA-LSTM ('+SA_MODE+')']
    a,b=traces[(ratio,names[0])],traces[(ratio,names[1])]
    ax.plot(a['date'],a['true'],label='Copper actual',color='black',linewidth=1)
    ax.plot(a['date'],a['pred'],label='LSTM baseline',alpha=.85)
    ax.plot(b['date'],b['pred'],label='SA-LSTM',alpha=.85)
    ax.set(title=f'Test {1-ratio:.1%} | next-day copper close',ylabel='USD per lb'); ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

fig,axes=plt.subplots(1,2,figsize=(12,4))
for name in ['LSTM baseline','SA-LSTM ('+SA_MODE+')']:
    h=traces[(0.90,name)]['history']
    axes[0].plot(h['loss'],label=name); axes[1].plot(h['val_loss'],label=name)
axes[0].set_title('Training MSE (scaled)');axes[1].set_title('Validation MSE (scaled)')
for ax in axes: ax.legend();ax.set_xlabel('Epoch');ax.grid(alpha=.2)
plt.tight_layout();plt.show()

## 6. Bảng đối chiếu nghiên cứu và giới hạn

| Nguồn | Phương pháp | Kết quả | Phạm vi |
|---|---|---|---|
| Chen et al. 2023 | SA-LSTM | MSE **0.00195 / 0.00190 / 0.00097** | 485 / 363 / 242 phiên kiểm tra; giai đoạn 1990–2009, nguồn Investing.com; cách chuẩn hóa không đủ chi tiết để khẳng định cùng thang đo |
| [26] Dehghani & Bogdanovic (2018) | BA | **Không có số MSE được Chen et al. công bố trong bảng so sánh trực tiếp** | Kết quả tham khảo từ tài liệu [26], **không chạy trên cùng bộ dữ liệu hiện tại** |
| [27] Astudillo et al. (2020) | SVR | **Không có số MSE được Chen et al. công bố trong bảng so sánh trực tiếp** | Kết quả tham khảo từ tài liệu [27], **không chạy trên cùng bộ dữ liệu hiện tại** |
| [28] Khoshalan et al. (2021) | ANFIS | **Không có số MSE được Chen et al. công bố trong bảng so sánh trực tiếp** | Kết quả tham khảo từ tài liệu [28], **không chạy trên cùng bộ dữ liệu hiện tại** |

**Không bịa số BA/SVR/ANFIS:** cần đọc trực tiếp [26]–[28] và chuẩn hóa metric trước khi bổ sung trị số; bài PDF Chen chỉ trích dẫn công trình chứ không cho bảng số liệu đó. Các hàng LSTM và SA-LSTM ở bảng phía trên mới là kết quả **chạy cùng dữ liệu 2006–2026**.

**Khác biệt với bài gốc:** Yahoo thay Investing; giai đoạn khác; lookback 30 do notebook gốc (không được nêu rõ trong bài); cách validation riêng để tránh data leakage; feature selection theo TRAIN thay vì tính toàn bộ 1990–2009. Vì vậy đây là *methodology replication on updated data*, không phải exact reproduction of published MSE.

**Tài liệu:** Chen, J. et al. (2023), *PLOS ONE* 18(10), e0285631, DOI 10.1371/journal.pone.0285631; [26] *Resources Policy* 55, 55–61; [27] *Applied Sciences* 10(19), 6648; [28] *Resources Policy* 73, 102239.